# Algoritmo de Shor

El algoritmo de Shor factoriza un entero N en un tiempo que crece de forma polinómica con el número de bits de N, algo que no consigue ningún algoritmo clásico conocido. Por eso importa: la seguridad de RSA descansa en que factorizar números grandes es inabordable, y un ordenador cuántico lo bastante grande podría hacerlo. Este notebook factoriza 15 y 21 con Polypus, primero paso a paso y después con una sola llamada.

## La idea: factorizar es encontrar un orden

Se elige una base a sin factores comunes con N. Su **orden** r es el menor entero positivo con a^r ≡ 1 (mod N). Si r es par y a^(r/2) no es ≡ −1 (mod N), N divide a (a^(r/2) − 1)(a^(r/2) + 1) sin dividir a ninguno de los dos, así que mcd(a^(r/2) − 1, N) y mcd(a^(r/2) + 1, N) son divisores de N. Con N = 15 y a = 7, a mano:

In [ ]:
import math

N, a = 15, 7
print([pow(a, k, N) for k in range(1, 5)])  # 7, 4, 13, 1: el orden es r = 4

x = pow(a, 4 // 2, N)  # 7^2 mod 15 = 4
print(math.gcd(x - 1, N), math.gcd(x + 1, N))  # 3 y 5

Probar potencias una a una cuesta del orden de N pasos, una cantidad exponencial en el número de bits. La parte cuántica del algoritmo encuentra r con un circuito de tamaño polinómico; el resto es clásico.

## El circuito

`polypus.circuits.templates.shor_order_finding(a, N)` construye ese circuito de búsqueda del orden, sin medidas:

In [ ]:
from polypus.circuits.templates import shor_order_finding

qc = shor_order_finding(7, 15)
print(repr(qc))

Son 18 qubits y 8561 puertas, demasiadas para dibujarlas. Con n = 4 bits para N = 15, los qubits se reparten así:

- **0 a 7, el registro de conteo**, de t = 2n = 8 qubits. Empiezan en superposición con H, y el qubit k controla una multiplicación del registro de trabajo por a^(2^k) mod N. Al final, la transformada de [`04b`](04b_qft.ipynb), invertida, convierte las fases acumuladas en un número que se puede medir.
- **8 a 11, el registro de trabajo**, de n qubits. Empieza en 1 y cada multiplicación controlada lo multiplica.
- **12 a 16, un registro auxiliar** de n + 1 qubits, y **17, un ancilla**. Los usa la multiplicación modular, y vuelven a 0.

## Ejecutarlo

Basta con medir el registro de conteo, cada qubit i en el bit clásico i, y ejecutar el circuito en el simulador nativo:

In [ ]:
import polypus

for i in range(8):
    qc.measure(i, i)

result = polypus.run_quantum_circuit(
    qc, shots=1024, infrastructure="local", backend="polypus", seed=7
)
counts = result.merged_counts
print(sorted((int(bits, 2), n) for bits, n in counts.items()))

In [ ]:
import matplotlib.pyplot as plt

plt.bar([int(bits, 2) for bits in counts], list(counts.values()), width=6)
plt.xlabel("Valor medido y")
plt.ylabel("Número de veces")
plt.title("Registro de conteo de shor_order_finding(7, 15)")
plt.show()

Leído como un entero, con el qubit 0 como bit menos significativo, el registro de conteo solo da cuatro valores, con la misma frecuencia: 0, 64, 128 y 192. Son los múltiplos de 2^t / r = 256 / 4 = 64, es decir, y = s · 2^t / r con s = 0, 1, 2 y 3. Cada medida aproxima una fracción s / r, aquí de forma exacta porque r = 4 divide a 2^8.

## Del resultado al orden

`polypus.shor.order_from_measurement(y, t, a, N)` desarrolla y / 2^t en fracción continua y prueba los denominadores de sus aproximaciones, y algunos de sus múltiplos, hasta dar con un r que cumpla a^r ≡ 1 (mod N). Después, `factors_from_order(a, N, r)` hace el paso de los mcd:

In [ ]:
from polypus.shor import factors_from_order, order_from_measurement

print(order_from_measurement(64, 8, 7, 15))  # 64/256 = 1/4: r = 4
print(order_from_measurement(128, 8, 7, 15))  # 128/256 = 1/2: prueba 2, y después 4
print(order_from_measurement(0, 8, 7, 15))  # 0/256 no dice nada de r
print(factors_from_order(7, 15, 4))

El pico en 0 no sirve: corresponde a s = 0, y 0 / 2^t vale 0 / r para cualquier r. Sale en una de cada r medidas, así que basta con pasar al siguiente resultado más frecuente. El pico en 128 da la fracción 1/2, cuyo denominador divide al orden pero no es el orden: por eso se prueban también sus múltiplos.

## Todo junto

`polypus.shor.factor` hace el recorrido entero. Resuelve sin circuito los casos que no lo necesitan (un N par, una potencia de un entero, una base con un factor común con N); si no, elige una base al azar, ejecuta el circuito, procesa las medidas y, si no salen factores, lo intenta con otra base. Con `seed` el resultado es reproducible:

In [ ]:
from polypus.shor import factor

result = factor(15, infrastructure="local", backend="polypus", seed=7)
print(result)
assert result.factors == (3, 5)
assert sorted(int(bits, 2) for bits in result.counts) == [0, 64, 128, 192]

`method="quantum"` indica que el orden se leyó en los recuentos de una ejecución del circuito, y `attempts` cuenta esas ejecuciones. No demuestra que el circuito funcionara: la lectura prueba todos los resultados y varios múltiplos de cada orden, así que, con los N que se pueden simular, encuentra el orden incluso en recuentos sin señal. La prueba está en `counts`: aquí, con a = 7, son los cuatro picos de antes, 0, 64, 128 y 192. Con N = 21 y a = 2, cuyo orden es 6, el registro de conteo puede quedarse en 6 qubits en vez de 2n = 10: 18 qubits en total en vez de 22. Como 6 no divide a 2^6 = 64, los picos ya no son exactos y se reparten entre los valores cercanos a 64 · s / 6, pero la fracción continua sigue encontrando el orden:

In [ ]:
result = factor(
    21, a=2, counting_qubits=6, infrastructure="local", backend="polypus", seed=1
)
print(result.factors, result.method, result.order)
assert result.factors == (3, 7)

## Hasta dónde llega

El circuito tiene t + 2n + 2 qubits; con el registro de conteo por defecto, t = 2n, son 4n + 2. Los simuladores locales llegan a 30 qubits, que alcanzan para N < 128, de hasta 7 bits, pero cada qubit de más duplica el tiempo y la memoria de una ejecución: en cuatro núcleos, un intento tarda unos segundos para N < 16, alrededor de un minuto para N < 32, media hora para N < 64 (26 qubits, 1 GiB) y, con 30 qubits, necesita 16 GiB y muchas horas. La plantilla construye circuitos de hasta 24 bits, que sirven para estimar recursos aunque no se puedan simular. Por ejemplo, para un N de 16 bits:

In [ ]:
big = shor_order_finding(2, 251 * 257)  # N = 64507, de 16 bits
print(repr(big))

Son 66 qubits y unas 849 000 puertas. `to_qasm3` exporta estos circuitos hasta 16 bits; a partir de 17, el texto pasaría de los 32 MiB que lee el importador de OpenQASM 3. Un `counting_qubits` menor ahorra qubits y abarata cada ejecución, a cambio de recuentos menos precisos.

## Resumen

Este notebook ha factorizado 15 y 21 con el algoritmo de Shor: la búsqueda del orden de a módulo N con `shor_order_finding`, la lectura del orden con fracciones continuas con `order_from_measurement`, los mcd que dan los factores con `factors_from_order`, y todo junto con `polypus.shor.factor`.

La siguiente sección cubre los algoritmos de búsqueda con oráculo: Deutsch-Jozsa y Grover.

## Siguiente paso

Continuar con: [`05a_deutsch_jozsa.ipynb`](05a_deutsch_jozsa.ipynb).